In [1]:
import imaplib
import email
from email.header import decode_header
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from bs4 import BeautifulSoup
import sys
!{sys.executable} -m pip install pandas beautifulsoup4

Defaulting to user installation because normal site-packages is not writeable


In [28]:
import os

print("Email:", os.environ.get("GMAIL_USER"))
print("Password status:", "Loaded" if os.environ.get("GMAIL_APP_PASSWORD") else "Not found")

Email: chaitrajummai@gmail.com
Password status: Loaded


In [29]:
USERNAME = os.environ.get("GMAIL_USER")
APP_PASSWORD = os.environ.get("GMAIL_APP_PASSWORD")

In [1]:
import imaplib
imap_host = 'imap.gmail.com'

mail = imaplib.IMAP4_SSL(imap_host)
mail.login(USERNAME, APP_PASSWORD)
mail.select("inbox")

print("✅ Connected to Gmail inbox successfully!")

NameError: name 'USERNAME' is not defined

In [5]:
import imaplib
import email
from email.header import decode_header
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm  # for progress bar

# Make sure you're already connected: mail = imaplib.IMAP4_SSL("imap.gmail.com")
mail.select("inbox")

# Search for all email IDs
status, data = mail.search(None, "ALL")
mail_ids = data[0].split()
print(f"Total emails found: {len(mail_ids)}")

emails = []

# Loop through all emails with progress bar
for i in tqdm(mail_ids, desc="Fetching emails"):
    status, msg_data = mail.fetch(i, "(RFC822)")
    for response_part in msg_data:
        if isinstance(response_part, tuple):
            msg = email.message_from_bytes(response_part[1])
            
            # Decode subject safely
            subject, encoding = decode_header(msg["Subject"])[0]
            if isinstance(subject, bytes):
                subject = subject.decode(encoding if encoding else "utf-8", errors="ignore")
            
            from_ = msg.get("From")
            date_ = msg.get("Date")

            # Extract body (text/plain or text/html)
            body = ""
            if msg.is_multipart():
                for part in msg.walk():
                    content_type = part.get_content_type()
                    content_disposition = str(part.get("Content-Disposition"))
                    try:
                        if content_type == "text/plain" and "attachment" not in content_disposition:
                            body = part.get_payload(decode=True).decode(errors="ignore")
                            break
                        elif content_type == "text/html" and not body:
                            html = part.get_payload(decode=True).decode(errors="ignore")
                            soup = BeautifulSoup(html, "lxml")
                            body = soup.get_text()
                    except Exception:
                        pass
            else:
                body = msg.get_payload(decode=True).decode(errors="ignore")

            emails.append({
                "Subject": subject,
                "From": from_,
                "Date": date_,
                "Body": body.strip()
            })

# Store all emails into DataFrame
df = pd.DataFrame(emails)

# Show top 5 emails

pd.set_option('display.max_rows', None)     # show all rows
pd.set_option('display.max_colwidth', None) # show full email content


Total emails found: 448


Fetching emails: 100%|███████████████████████████████████████████████████████████████| 448/448 [03:58<00:00,  1.88it/s]


In [6]:
import pandas as pd

# Suppose df contains all fetched emails
df.to_csv("fetched_emails.csv", index=False)
print("✅ Emails saved to fetched_emails.csv")


✅ Emails saved to fetched_emails.csv


In [7]:
# Check if df exists
try:
    print("✅ df found! Columns:", df.columns.tolist())
except NameError:
    print("⚠️ 'df' not found. Please re-run your email fetching code first.")

✅ df found! Columns: ['Subject', 'From', 'Date', 'Body']


In [10]:
# ✅ Data cleaning and formatted display for ALL emails
import pandas as pd
import re
from bs4 import BeautifulSoup
from urllib.parse import urlparse
import html

# -------- CLEANING FUNCTION --------
def domain_token(host):
    if not host:
        return ""
    parts = host.split('.')
    parts = [p for p in parts if p not in ('www','m','mobile')]
    if not parts:
        return ""
    token = parts[-2] if len(parts) >= 2 else parts[-1]
    token = re.sub(r'[^A-Za-z]', '', token)
    return token.lower()

URL_RE = re.compile(r'(?:(?:https?|ftp):\/\/[^\s<>"]+|www\.[^\s<>"]+|<https?:\/\/[^>]+>|mailto:[^\s<>"]+)', re.IGNORECASE)
LONG_GARBAGE_RE = re.compile(r'[A-Za-z0-9+/=_-]{25,}')
ANGLE_BRACKET_RE = re.compile(r'<[^>]{3,}>')

def clean_email_keep_domains(text):
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    text = text.replace('\\r\\n', '\n').replace('\\r', '\n').replace('\\n', '\n')
    if ('<' in text and '>' in text):
        soup = BeautifulSoup(text, "html.parser")
        for tag in soup(['script','style']):
            tag.decompose()
        text = soup.get_text(separator=' ')
    def url_repl(match):
        url = match.group(0)
        url_clean = url.strip('<>')
        try:
            if url_clean.lower().startswith('mailto:'):
                return ' '
            parsed = urlparse(url_clean if '://' in url_clean else 'http://' + url_clean)
            host = parsed.hostname or ''
            token = domain_token(host)
            return (' ' + token + ' ') if token else ' '
        except Exception:
            return ' '
    text = URL_RE.sub(url_repl, text)
    text = ANGLE_BRACKET_RE.sub(' ', text)
    text = LONG_GARBAGE_RE.sub(' ', text)
    text = re.sub(r'\b\S+@\S+\b', ' ', text)
    text = re.sub(r'[^\w\s\'-]', ' ', text)
    text = re.sub(r'\s{2,}', ' ', text).strip()
    return text

# -------- APPLY CLEANING TO ALL EMAILS --------
if 'df' not in globals():
    raise NameError("Run your email fetching cell first (the one that creates df).")

df['Cleaned_Body'] = df['Body'].apply(clean_email_keep_domains)

# -------- NEAT TABLE DISPLAY --------
pd.set_option('display.max_rows', 1)          # Show all rows
pd.set_option('display.max_colwidth', 200)       # Limit column width for readability
pd.set_option('display.expand_frame_repr', False)

display(
    df[['Subject', 'From', 'Date', 'Cleaned_Body']]
)

,Subject,From,Date,Cleaned_Body
0,"Chaitra, finish setting up your new Google Account",Google Community Team <googlecommunityteam-noreply@google.com>,"Thu, 16 Feb 2023 00:26:44 -0800",Hi Chaitra Welcome to Google Your new account comes with access to Google products apps and services Here are a few tips to get you started Get the most out of your Google Account We'll send you p...


In [11]:
import pandas as pd

# Save the cleaned dataframe (df) permanently

df.to_csv("cleaned_emails.csv", index=False)

print("✅ Cleaned emails saved successfully to cleaned_emails.csv")



✅ Cleaned emails saved successfully to cleaned_emails.csv


In [1]:
#run 1st this in notebook
import pandas as pd

# Load your previously saved data
df = pd.read_csv("cleaned_emails.csv")

# 🟢 Display confirmation message
print("\n✅ Cleaned emails loaded successfully!\n")
print("🧾 Total Emails Loaded:", len(df))
print("\n📬 Displaying all cleaned emails:\n")

# 🧹 Optional: Show all rows & wrap text neatly
pd.set_option('display.max_rows', 1)            # ✅ Show only 1 row
pd.set_option('display.max_colwidth', None)     # Don’t truncate long text
pd.set_option('display.expand_frame_repr', False)

# 📊 Display selected columns if available
if {'From', 'Subject', 'Cleaned_Body'}.issubset(df.columns):
    display(df[['From', 'Subject', 'Cleaned_Body']].head(1))   # ✅ Show only 1 email
else:
    display(df.head(1))                                        # ✅ Show only 1 email

print("\n✅ All cleaned emails displayed successfully.")


FileNotFoundError: [Errno 2] No such file or directory: 'cleaned_emails.csv'

In [13]:
import pandas as pd

# Save the cleaned dataframe (df) permanently

df.to_csv("cleaned_emails.csv", index=False)

print("✅ Cleaned emails saved successfully to cleaned_emails.csv")



✅ Cleaned emails saved successfully to cleaned_emails.csv


In [14]:
# ✅ Load cleaned emails safely and show only 1 sample
import pandas as pd

# Load your previously saved data
df = pd.read_csv("cleaned_emails.csv")

# 🟢 Confirmation message
print("\n✅ Cleaned emails loaded successfully!\n")
print("🧾 Total Emails Loaded:", len(df))

# 📊 Display only one cleaned email (memory-safe)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)

print("\n📬 Showing 1 sample cleaned email:\n")

if {'From', 'Subject', 'Cleaned_Body'}.issubset(df.columns):
    display(df[['From', 'Subject', 'Cleaned_Body']].head(1))
else:
    display(df.head(1))

print("\n✅ Display complete — only 1 cleaned email shown.")



✅ Cleaned emails loaded successfully!

🧾 Total Emails Loaded: 448

📬 Showing 1 sample cleaned email:



,From,Subject,Cleaned_Body
0,Google Community Team <googlecommunityteam-noreply@google.com>,"Chaitra, finish setting up your new Google Account",Hi Chaitra Welcome to Google Your new account comes with access to Google products apps and services Here are a few tips to get you started Get the most out of your Google Account We'll send you personalized tips news and recommendations from Google Yes keep me updated Stay in the know with the Google app Find quick answers explore your interests and stay up to date Try it More from Google Discover the latest apps from Google For Android For iOS Confirm your options are right for you Review and change your privacy and security options to make Google work better for you Confirm Find answers Visit the Help Center to learn all about your new Google Account Replies to this email aren't monitored If you have a question about your new account the Help Center likely has the answer you're looking for Google LLC 1600 Amphitheatre Parkway Mountain View CA 94043 This email was sent to you because you created a Google Account



✅ Display complete — only 1 cleaned email shown.


In [15]:
#deep clean
import pandas as pd
import re
from bs4 import BeautifulSoup

# Load your previously saved cleaned data
df = pd.read_csv("cleaned_emails.csv")

# -------- Deep Cleaning Function --------
def deep_clean(text):
    text = str(text)

    # 1️⃣ Remove HTML tags
    text = BeautifulSoup(text, "html.parser").get_text()

    # 2️⃣ Remove URLs (http, https, www)
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # 3️⃣ Remove common encoded entities (&nbsp;, &amp;, etc.)
    text = re.sub(r'&[a-z]+;', ' ', text)

    # 4️⃣ Remove excessive punctuation and whitespace
    text = re.sub(r'[^A-Za-z0-9\s.,!?@%$]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply deep cleaning to every email
df['Cleaned_Body_Final'] = df['Cleaned_Body'].apply(deep_clean)

# Save to new CSV
df.to_csv("cleaned_emails_final.csv", index=False)

# -------- Clean Output Display --------
print("\n✅ Saved successfully as: cleaned_emails_final.csv")
print(f"📩 Total emails processed: {len(df)}\n")

print("🧹 Showing 2 sample cleaned emails:\n")

# ✅ Display options tuned for clarity
pd.set_option('display.max_rows', 2)         # force exactly 2 rows visible
pd.set_option('display.max_colwidth', 150)   # shorter column width for wrapping
pd.set_option('display.width', 0)            # let pandas use full width
pd.set_option('display.expand_frame_repr', False)

display(df[['From', 'Subject', 'Cleaned_Body_Final']].head(2))

print("\n✅ Deep cleaning completed and both samples displayed neatly.")



✅ Saved successfully as: cleaned_emails_final.csv
📩 Total emails processed: 448

🧹 Showing 2 sample cleaned emails:



,From,Subject,Cleaned_Body_Final
0,Google Community Team <googlecommunityteam-noreply@google.com>,"Chaitra, finish setting up your new Google Account",Hi Chaitra Welcome to Google Your new account comes with access to Google products apps and services Here are a few tips to get you started Get th...
1,no-reply.jeemain.nta@nic.in,NTA - JEE(Main) 2023 - Session 2 0012120 Application Enrollment -\r\n Completed,Dear CHAITRA APPASAB JUMMAI Thanks for applying for JEE Main 2023 Session 2 Your Application Number is 230320012120 To complete application form s...



✅ Deep cleaning completed and both samples displayed neatly.


In [16]:
import pandas as pd

df = pd.read_csv("cleaned_emails_final.csv")

def preview(text, limit=40):
    words = str(text).split()
    return " ".join(words[:limit]) + ("..." if len(words) > limit else "")

df['Preview'] = df['Cleaned_Body_Final'].apply(lambda t: preview(t))
df['Category'] = ""  # empty column for labeling

cols = ['From', 'Subject', 'Preview', 'Category']
df[cols].to_csv("emails_for_manual_labeling.csv", index=False)
print("✅ File ready for labeling: emails_for_manual_labeling.csv")


✅ File ready for labeling: emails_for_manual_labeling.csv


In [1]:
import pandas as pd
import os

base_path = r"C:\Users\DELL\Desktop\emaildataset"

files = [
    "Academic_csv.csv",
    "administration_dataset.csv",
    "campus_service_90.csv",
    "career_opportunity_33.csv",
    "financial_services_13.csv",
    "skill_development_46.csv",
    "service_csv.csv",
    "scholarship_dataset_90.csv",
    "miscellaneous_dataset_94.csv"
]

for f in files:
    full_path = os.path.join(base_path, f)
    df = pd.read_csv(full_path)
    
    print(f"{f}: {len(df)} rows")

Academic_csv.csv: 100 rows
administration_dataset.csv: 100 rows
campus_service_90.csv: 100 rows
career_opportunity_33.csv: 100 rows
financial_services_13.csv: 100 rows
skill_development_46.csv: 100 rows
service_csv.csv: 100 rows
scholarship_dataset_90.csv: 100 rows
miscellaneous_dataset_94.csv: 100 rows


In [12]:
# S1: Split datasets into train + test for each category

import pandas as pd
import os

base_path = r"C:\Users\DELL\Desktop\emaildataset"

files = [
    "Academic_csv.csv",
    "administration_dataset.csv",
    "campus_service_90.csv",
    "career_opportunity_33.csv",
    "financial_services_13.csv",
    "skill_development_46.csv",
    "service_csv.csv",
    "scholarship_dataset_90.csv",
    "miscellaneous_dataset_94.csv"
]

train_folder = os.path.join(base_path, "train_manual")
test_folder  = os.path.join(base_path, "test_manual")

os.makedirs(train_folder, exist_ok=True)
os.makedirs(test_folder, exist_ok=True)

for f in files:
    full_path = os.path.join(base_path, f)
    df = pd.read_csv(full_path)

    df = df.dropna(how='all')  # remove empty rows
    df = df.dropna(subset=["From", "Subject", "Preview"], how='any')

    df = df.head(100)  # ensure exactly 100 rows

    test_df = df.head(20)   # first 20 rows
    train_df = df.tail(80)  # last 80 rows

    train_df.to_csv(os.path.join(train_folder, "train_" + f), index=False)
    test_df.to_csv(os.path.join(test_folder,  "test_"  + f), index=False)

    print(f"Done: {f} → Train={len(train_df)}, Test={len(test_df)}")


Done: Academic_csv.csv → Train=80, Test=20
Done: administration_dataset.csv → Train=80, Test=20
Done: campus_service_90.csv → Train=80, Test=20
Done: career_opportunity_33.csv → Train=80, Test=20
Done: financial_services_13.csv → Train=80, Test=20
Done: skill_development_46.csv → Train=80, Test=20
Done: service_csv.csv → Train=80, Test=20
Done: scholarship_dataset_90.csv → Train=80, Test=20
Done: miscellaneous_dataset_94.csv → Train=80, Test=20


In [13]:
import pandas as pd
import os

base_path = r"C:\Users\DELL\Desktop\emaildataset"

train_folder = os.path.join(base_path, "train_manual")
test_folder  = os.path.join(base_path, "test_manual")

# Combine TRAIN files (9×80=720)
train_dfs = []
for f in os.listdir(train_folder):
    if f.endswith(".csv"):
        train_dfs.append(pd.read_csv(os.path.join(train_folder, f)))

train_final = pd.concat(train_dfs, ignore_index=True)
train_final.to_csv(os.path.join(base_path, "train_final.csv"), index=False)

print("Train shape:", train_final.shape)

# Combine TEST files (9×20=180)
test_dfs = []
for f in os.listdir(test_folder):
    if f.endswith(".csv"):
        test_dfs.append(pd.read_csv(os.path.join(test_folder, f)))

test_final = pd.concat(test_dfs, ignore_index=True)
test_final.to_csv(os.path.join(base_path, "test_final.csv"), index=False)

print("Test shape:", test_final.shape)


Train shape: (720, 4)
Test shape: (180, 4)


In [14]:
# S1.4: Prepare final training files with 6 columns
import pandas as pd

# Load Step 3 results (which have 4 columns)
train = pd.read_csv(r"C:\Users\DELL\Desktop\emaildataset\train_final.csv")
test  = pd.read_csv(r"C:\Users\DELL\Desktop\emaildataset\test_final.csv")

# Replace NaN with empty string
train = train.fillna("")
test = test.fillna("")

# Add the missing Domain column  (FIXED VERSION)
train['Domain'] = train['From'].astype(str).str.split('@').str[-1]
test['Domain']  = test['From'].astype(str).str.split('@').str[-1]

# Add the missing text column
train['text'] = train['Subject'] + " " + train['Preview'] + " " + train['Domain']
test['text']  = test['Subject'] + " " + test['Preview'] + " " + test['Domain']

# Save the new files with 6 columns
train.to_csv(r"C:\Users\DELL\Desktop\emaildataset\train_prepared.csv", index=False)
test.to_csv(r"C:\Users\DELL\Desktop\emaildataset\test_prepared.csv", index=False)

print(train.shape)  
print(test.shape)

(720, 6)
(180, 6)


In [15]:
import pandas as pd
import os

base_path = r"C:\Users\DELL\Desktop\emaildataset"

train_folder = os.path.join(base_path, "train_manual")
test_folder  = os.path.join(base_path, "test_manual")

# Combine TRAIN files (9×80=720)
train_dfs = []
for f in os.listdir(train_folder):
    if f.endswith(".csv"):
        train_dfs.append(pd.read_csv(os.path.join(train_folder, f)))

train_final = pd.concat(train_dfs, ignore_index=True)
train_final.to_csv(os.path.join(base_path, "train_final.csv"), index=False)

print("Train shape:", train_final.shape)

# Combine TEST files (9×20=180)
test_dfs = []
for f in os.listdir(test_folder):
    if f.endswith(".csv"):
        test_dfs.append(pd.read_csv(os.path.join(test_folder, f)))

test_final = pd.concat(test_dfs, ignore_index=True)
test_final.to_csv(os.path.join(base_path, "test_final.csv"), index=False)

print("Test shape:", test_final.shape)


Train shape: (720, 4)
Test shape: (180, 4)


In [16]:
# S3: Add Domain and text columns

train = pd.read_csv(base_path + "\\train_final.csv")
test  = pd.read_csv(base_path + "\\test_final.csv")

train = train.fillna("")
test = test.fillna("")

train["Domain"] = train["From"].astype(str).str.split("@").str[-1]
test["Domain"]  = test["From"].astype(str).str.split("@").str[-1]

train["text"] = train["Subject"] + " " + train["Preview"] + " " + train["Domain"]
test["text"]  = test["Subject"] + " " + test["Preview"] + " " + test["Domain"]

train.to_csv(base_path + "\\train_prepared.csv", index=False)
test.to_csv(base_path + "\\test_prepared.csv", index=False)

print(train.shape)
print(test.shape)


(720, 6)
(180, 6)


In [18]:
import pandas as pd

# Load prepared training and test files
train = pd.read_csv(r"C:\Users\DELL\Desktop\emaildataset\train_prepared.csv")
test  = pd.read_csv(r"C:\Users\DELL\Desktop\emaildataset\test_prepared.csv")

# Extract features (text) and labels (Category)
X_train = train['text']
y_train = train['Category']

X_test  = test['text']
y_test  = test['Category']

print("Train samples:", len(X_train))
print("Test samples:", len(X_test))

Train samples: 720
Test samples: 180


In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english', max_features=5000)

X_train_tf = tfidf.fit_transform(X_train)
X_test_tf  = tfidf.transform(X_test)

print("Training TF-IDF:", X_train_tf.shape)
print("Testing TF-IDF:", X_test_tf.shape)


Training TF-IDF: (720, 2706)
Testing TF-IDF: (180, 2706)


In [20]:
from sklearn.svm import LinearSVC

model = LinearSVC()
model.fit(X_train_tf, y_train)

print("Model training completed.")


Model training completed.


In [32]:
import warnings
warnings.filterwarnings("ignore")
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd

y_pred = model.predict(X_test_tf)

y_test_clean = y_test.astype(str)
y_pred_clean = pd.Series(y_pred).astype(str)

# Accuracy
print("\n⭐ Accuracy:")
print(accuracy_score(y_test_clean, y_pred_clean))

# Classification report
print("\n⭐ Classification Report:\n")
print(classification_report(
    y_test_clean,
    y_pred_clean,
    digits=4,
    zero_division=0
))

# Confusion matrix
labels = sorted(test['Category'].astype(str).unique())
cm = confusion_matrix(y_test_clean, y_pred_clean, labels=labels)

cm_df = pd.DataFrame(
    cm,
    index=[f"True: {label}" for label in labels],
    columns=[f"Pred: {label}" for label in labels]
)

cm_df



⭐ Accuracy:
0.9611111111111111

⭐ Classification Report:

                      precision    recall  f1-score   support

            Academic     1.0000    0.8000    0.8889        20
      Administration     0.8571    0.9000    0.8780        20
      Campus Service     0.8696    1.0000    0.9302        20
  Career/Opportunity     1.0000    1.0000    1.0000        20
  Financial Services     0.9524    1.0000    0.9756        20
       Miscellaneous     1.0000    1.0000    1.0000        20
        Scholarship      1.0000    1.0000    1.0000        20
Service Notification     1.0000    0.9500    0.9744        20
   Skill Development     1.0000    1.0000    1.0000        20

            accuracy                         0.9611       180
           macro avg     0.9643    0.9611    0.9608       180
        weighted avg     0.9643    0.9611    0.9608       180



,Pred: Academic,Pred: Administration,Pred: Campus Service,Pred: Career/Opportunity,Pred: Financial Services,Pred: Miscellaneous,Pred: Scholarship,Pred: Service Notification,Pred: Skill Development
True: Academic,16,3,1,0,0,0,0,0,0
True: Administration,0,18,2,0,0,0,0,0,0
True: Campus Service,0,0,20,0,0,0,0,0,0
True: Career/Opportunity,0,0,0,20,0,0,0,0,0
True: Financial Services,0,0,0,0,20,0,0,0,0
True: Miscellaneous,0,0,0,0,0,20,0,0,0
True: Scholarship,0,0,0,0,0,0,20,0,0
True: Service Notification,0,0,0,0,1,0,0,19,0
True: Skill Development,0,0,0,0,0,0,0,0,20


In [22]:
results = test.copy()
results["Predicted_Category"] = y_pred

results = results[["From", "Subject", "Preview", "Domain",
                   "Category", "Predicted_Category"]]

output_path = r"C:\Users\DELL\Desktop\emaildataset\model_classification_results.csv"
results.to_csv(output_path, index=False)

print("✔ CSV generated successfully!")
print("Saved at:", output_path)


✔ CSV generated successfully!
Saved at: C:\Users\DELL\Desktop\emaildataset\model_classification_results.csv


In [25]:
# Compare true vs predicted
results["Correct"] = results["Category"] == results["Predicted_Category"]

total = len(results)
correct = results["Correct"].sum()
wrong = total - correct

print("✔ Total Test Emails:", total)
print("✔ Correctly Classified:", correct)
print("❌ Misclassified:", wrong)

# Show wrongly classified rows
misclassified = results[results["Correct"] == False]




✔ Total Test Emails: 180
✔ Correctly Classified: 173
❌ Misclassified: 7


In [26]:
import pickle

pickle.dump(model, open("best_model.sav", "wb"))
pickle.dump(tfidf, open("tfidf_vectorizer.sav", "wb"))
